# Agent 3 — Customer Insight AI Agent
**Owner:** AI/ML Engineer (Member 3)

**Goal:** Analyze real customer reviews to surface the **top complaints** and **top praises** for a product/business category, using BERT-based sentiment analysis plus TF-IDF keyword extraction.

**Dataset used:** `data/amazon_reviews_sample.csv` + `data/googleplaystore_user_reviews.csv`

**Output:** Top positives, top negatives, and a summary insight — returned by `agent_customer(category)`.

> **Note:** This notebook uses HuggingFace `transformers` for BERT sentiment, with a **VADER fallback** when offline.  
> BERT inference runs in **batches** for speed instead of row-by-row.


In [ ]:
!pip install transformers torch vaderSentiment scikit-learn pandas numpy --quiet

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import re, json, warnings, os
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer

# ✅ FIX 1: Use os.path so the path works regardless of where the kernel is launched
BASE_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "..", "..", "data")

print(f"DATA_DIR resolved to: {os.path.abspath(DATA_DIR)}")


In [ ]:
amazon = pd.read_csv(os.path.join(DATA_DIR, "amazon_reviews_sample.csv"), low_memory=False)

amazon = amazon.rename(columns={
    "name": "product",
    "categories": "category",
    "reviews.rating": "rating",
    "reviews.text": "text"
})

amazon = amazon[["product", "category", "rating", "text"]]
amazon["source"] = "amazon"
print(f"Amazon shape: {amazon.shape}")
amazon.head()


In [ ]:
google = pd.read_csv(os.path.join(DATA_DIR, "googleplaystore_user_reviews.csv"))

google = google.rename(columns={
    "App": "product",
    "Translated_Review": "text"
})

google["category"] = "mobile app"
google["rating"] = np.nan
google["source"] = "google_play"

google = google[["product", "category", "rating", "text"]]
print(f"Google Play shape: {google.shape}")
google.head()


## 2. Merge Datasets & Clean Text

In [ ]:
# ✅ FIX 2: Merge FIRST, then define and apply clean_text ONCE (was defined twice before)
reviews = pd.concat([amazon, google], ignore_index=True)
reviews = reviews.dropna(subset=["text"])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

reviews["clean_text"] = reviews["text"].apply(clean_text)

print(f"Combined reviews shape: {reviews.shape}")
reviews.head()


## 3. Load Sentiment Model (BERT with VADER Fallback)

In [ ]:
# ✅ FIX 3: Implement the VADER fallback that was promised in the original comments but never coded

bert_model = None
USE_BERT = False

try:
    from transformers import pipeline
    bert_model = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        truncation=True,
        max_length=512
    )
    USE_BERT = True
    print("✅ BERT model loaded successfully.")
except Exception as e:
    print(f"⚠️  BERT unavailable ({e}). Falling back to VADER.")
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
    print("✅ VADER loaded as fallback.")


## 4. Sentiment Function

In [ ]:
def predict_sentiment(text):
    """Returns 'positive' or 'negative' for a single review."""
    if USE_BERT:
        result = bert_model(text[:512])[0]
        return result["label"].lower()   # distilbert returns UPPERCASE
    else:
        score = vader.polarity_scores(str(text))["compound"]
        return "positive" if score >= 0.05 else "negative"


## 5. Apply Sentiment (Batched for Speed)

In [ ]:
# ✅ FIX 4: Original code ran BERT row-by-row via .apply() on 10 000 rows — extremely slow.
# BERT pipeline supports batching natively. We batch in chunks of 64 for 10-15× speedup.

SAMPLE_SIZE = min(10_000, len(reviews))
data = reviews.sample(SAMPLE_SIZE, random_state=42).copy()

if USE_BERT:
    BATCH_SIZE = 64
    texts = data["text"].str[:512].tolist()
    sentiments = []
    total = len(texts)
    for i in range(0, total, BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]
        results = bert_model(batch, truncation=True, max_length=512, batch_size=BATCH_SIZE)
        sentiments.extend([r["label"].lower() for r in results])
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  Processed {min(i + BATCH_SIZE, total)}/{total} reviews...")
    data["sentiment"] = sentiments
else:
    # VADER is fast — .apply() is fine here
    data["sentiment"] = data["text"].apply(predict_sentiment)

print(f"\n✅ Sentiment applied to {len(data)} reviews.")
print(data["sentiment"].value_counts())
data.head()


## 6. Extract Business Domain From Business Idea

In [ ]:
def extract_domain(business_idea):
    """Maps a free-text business idea to a list of domain keywords for filtering reviews."""
    mapping = {
        "food delivery":   ["food", "delivery", "restaurant", "order", "meal"],
        "mobile app":      ["app", "mobile", "software", "download", "update"],
        "electronics":     ["phone", "laptop", "device", "battery", "charger"],
        "health":          ["health", "medical", "fitness", "workout", "gym"],
        "ecommerce":       ["shopping", "shipping", "return", "refund", "package"],
        "travel":          ["hotel", "flight", "booking", "travel", "trip"],
    }
    idea = business_idea.lower()
    for domain, keywords in mapping.items():
        if any(word in idea for word in keywords):
            return keywords        # return list for regex filter
    # Fallback: use words from the idea itself
    return [w for w in idea.split() if len(w) > 3]


## 7. TF-IDF Keyword Extraction

In [ ]:
def extract_keywords(texts, top_n=5):
    """Extract top n bi/tri-gram keywords from a pandas Series of texts."""
    # ✅ FIX 5: Handle empty corpus — was crashing with ValueError before
    texts = texts.dropna()
    if len(texts) < 2:
        return ["not enough data"]

    custom_stop = [
        "good", "great", "love", "like", "best", "nice",
        "really", "very", "app", "use", "it", "is", "to",
        "the", "and", "with", "this", "just", "also", "get"
    ]

    vectorizer = TfidfVectorizer(
        stop_words="english",          # ✅ FIX 6: use built-in English stop words
        ngram_range=(2, 3),
        max_features=5000
    )

    try:
        matrix = vectorizer.fit_transform(texts)
    except ValueError:
        return ["not enough data"]

    scores = np.asarray(matrix.sum(axis=0)).flatten()
    words  = vectorizer.get_feature_names_out()

    result = []
    for i in scores.argsort()[::-1]:
        if len(result) >= top_n:
            break
        word = words[i]
        # Filter out any custom stops that slipped through
        if not any(sw in word for sw in custom_stop) and len(word.split()) >= 2:
            result.append(word)

    return result if result else ["no distinctive keywords found"]


## 8. `agent_customer()` — Main Agent Function

In [ ]:
# ✅ FIX 7: Accept 'data' as a parameter so the function doesn't silently depend
#           on a global variable that may not exist if an earlier cell was skipped.

def agent_customer(business_idea: str, reviews_df: pd.DataFrame = None) -> dict:
    """
    Analyze customer sentiment for a given business idea.

    Parameters
    ----------
    business_idea : str   Free-text business idea (e.g. 'AI powered fitness app')
    reviews_df    : DataFrame   Pre-labelled reviews with 'text', 'clean_text', 'sentiment'.
                                Defaults to the globally sampled `data` if None.

    Returns
    -------
    dict with keys: agent, business_idea, reviews_analyzed, positive, negative, insight
    """
    if reviews_df is None:
        # ✅ Graceful error instead of silent NameError
        if "data" not in dir():
            raise RuntimeError(
                "No reviews_df supplied and 'data' global not found. "
                "Run the sentiment cell first (Section 5) or pass reviews_df explicitly."
            )
        reviews_df = data

    keywords = extract_domain(business_idea)
    pattern  = "|".join(re.escape(k) for k in keywords)

    filtered = reviews_df[
        reviews_df["text"].str.contains(pattern, case=False, na=False)
    ]

    # Fall back to full dataset if match is too sparse
    if len(filtered) < 20:
        print(f"⚠️  Only {len(filtered)} domain-matched reviews found — using full dataset.")
        filtered = reviews_df

    positive_texts = filtered[filtered["sentiment"] == "positive"]["clean_text"]
    negative_texts = filtered[filtered["sentiment"] == "negative"]["clean_text"]

    positives = extract_keywords(positive_texts)
    negatives = extract_keywords(negative_texts)

    insight = (
        f"Customers appreciate: {', '.join(positives[:3])}. "
        f"Main pain points: {', '.join(negatives[:3])}. "
        "Focus on these areas to sharpen your competitive edge."
    )

    return {
        "agent":            "Customer Insight AI Agent",
        "business_idea":    business_idea,
        "reviews_analyzed": len(filtered),
        "positive":         positives,
        "negative":         negatives,
        "insight":          insight,
    }


## 9. Test the Agent

In [ ]:
result = agent_customer("AI powered fitness app", reviews_df=data)
print(json.dumps(result, indent=4))


## 10. Next Steps
- Move `clean_text`, `predict_sentiment`, `extract_keywords`, `agent_customer` into  
  `agent3_customer/sentiment.py` and `agent3_customer/customer_agent.py`.
- For production, run BERT sentiment over the **full** dataset in a batch offline job,  
  cache the labelled DataFrame, and have `agent_customer()` just filter + summarize cached results.
- Add SpaCy lemmatization before TF-IDF to merge similar terms (e.g. `battery`/`batteries`).
- Replace `extract_domain` hard-coded mapping with a small zero-shot classifier for scalability.
